In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
import koreanize_matplotlib
import warnings
warnings.filterwarnings('ignore')

In [15]:
df = pd.read_csv('../Data/당뇨_전처리.csv')
df

,임신횟수,혈당,혈압,피부두께,인슐린,BMI,가족력지표,나이,당뇨,Gluc_BMI,Age_Group
0,6,148.0,72.0,35.0,159.0,33.6,0.627,50,1,4972.8,2
1,1,85.0,66.0,29.0,95.0,26.6,0.351,31,0,2261.0,2
2,8,183.0,64.0,32.0,159.0,23.3,0.672,32,1,4263.9,2
3,1,89.0,66.0,23.0,94.0,28.1,0.167,21,0,2500.9,1
4,0,137.0,40.0,35.0,168.0,43.1,2.288,33,1,5904.7,2
...,...,...,...,...,...,...,...,...,...,...,...
713,10,101.0,76.0,48.0,180.0,32.9,0.171,63,0,3322.9,3
714,2,122.0,70.0,27.0,95.0,36.8,0.340,27,0,4489.6,1
715,5,121.0,72.0,23.0,112.0,26.2,0.245,30,0,3170.2,1
716,1,126.0,60.0,32.0,159.0,30.1,0.349,47,1,3792.6,2


In [16]:
df = df[['BMI','혈당','나이','가족력지표' ,'당뇨']]
# df = df[['혈당','나이','BMI','가족력지표','당뇨']]
# df = df[['혈당', 'BMI', '나이', '가족력지표','당뇨']]

In [17]:
df

,BMI,혈당,나이,가족력지표,당뇨
0,33.6,148.0,50,0.627,1
1,26.6,85.0,31,0.351,0
2,23.3,183.0,32,0.672,1
3,28.1,89.0,21,0.167,0
4,43.1,137.0,33,2.288,1
...,...,...,...,...,...
713,32.9,101.0,63,0.171,0
714,36.8,122.0,27,0.340,0
715,26.2,121.0,30,0.245,0
716,30.1,126.0,47,0.349,1


In [18]:
data = df.iloc[:,:-1]
target = df.iloc[:,-1]

train_data, test_data, train_target, test_target = train_test_split(
    data,
    target,
    random_state=42,
    test_size= 0.2,
    stratify=df.iloc[:,-1]
)

from sklearn.preprocessing import StandardScaler

ss = StandardScaler()
ss.fit(train_data)
train_scaled = ss.transform(train_data)
test_scaled = ss.transform(test_data)


In [19]:
print(train_data.shape)
print(test_data.shape)
print(train_target.shape)
print(test_target.shape)

(574, 4)
(144, 4)
(574,)
(144,)


In [20]:
x_train_data, x_valid_data, y_train_target, y_valid_target = train_test_split(
    train_scaled,
    train_target,
    random_state=42,
    test_size= 0.2,
    stratify=train_target
)

In [21]:
train_scaled

array([[ 1.68776653,  1.08208203,  0.26597507,  0.12437625],
       [ 1.01130464,  0.22815126,  1.38035238, -0.39593703],
       [-1.11032584, -0.42083613, -0.93412357, -1.04710753],
       ...,
       [-0.23400021,  1.56028327,  1.38035238, -0.5423725 ],
       [ 0.91905983,  0.91129588, -0.41979558, -0.44267175],
       [ 1.51865105,  1.08208203,  0.43741773,  0.82851278]])

In [22]:
depth_list = range(1, 31)  # 깊이 1 ~ 30
dt_cross_val_scores = []

for depth in depth_list:
    dt = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42
    )

    scores = cross_val_score(
        dt,
        x_train_data,
        y_train_target,
        cv=6,                  # 6-fold CV
        scoring='accuracy',
        n_jobs=-1
    )

    dt_cross_val_scores.append(scores.mean())

best_depth = depth_list[dt_cross_val_scores.index(max(dt_cross_val_scores))]
best_depth

5

In [23]:
dt = DecisionTreeClassifier(
    max_depth=best_depth,
    random_state=42
).fit(x_train_data, y_train_target)
xdtTrain = []
xdtValid = []
for i in range(5):
    xdtTrain.append(dt.score(x_train_data, y_train_target))
    xdtValid.append(dt.score(x_valid_data, y_valid_target))
    print(dt.score(x_train_data, y_train_target))
    print(dt.score(x_valid_data, y_valid_target))

0.8583877995642701
0.7304347826086957
0.8583877995642701
0.7304347826086957
0.8583877995642701
0.7304347826086957
0.8583877995642701
0.7304347826086957
0.8583877995642701
0.7304347826086957


In [24]:
print('xTrain : ',np.mean(xdtTrain))
print('Valid : ',np.mean(xdtValid))

xTrain :  0.8583877995642701
Valid :  0.7304347826086957


In [25]:
dt = DecisionTreeClassifier(
    max_depth=best_depth,
    random_state=42
).fit(train_scaled, train_target)
dtTest = []
dtTrain = []
for i in range(5):
    dtTrain.append(dt.score(train_scaled, train_target))
    dtTest.append(dt.score(test_scaled, test_target))
    print(dt.score(train_scaled, train_target))
    print(dt.score(test_scaled, test_target))

0.8466898954703833
0.7013888888888888
0.8466898954703833
0.7013888888888888
0.8466898954703833
0.7013888888888888
0.8466898954703833
0.7013888888888888
0.8466898954703833
0.7013888888888888


In [26]:
print('Train : ',np.mean(dtTrain))
print('Test : ',np.mean(dtTest))

Train :  0.8466898954703833
Test :  0.7013888888888888


CV AUC: 0.8342342868276903 +/- 0.032899869922963
CV ACC: 0.7527777777777778 +/- 0.04413186116358152
Test ACC: 0.7291666666666666
Test AUC: 0.7952473958333335
              precision    recall  f1-score   support

           0       0.79      0.80      0.80        96
           1       0.60      0.58      0.59        48

    accuracy                           0.73       144
   macro avg       0.69      0.69      0.69       144
weighted avg       0.73      0.73      0.73       144

